# Chapter 8.1 - Deep Convolutional Neural Networks (AlexNet)

Chapter 7 ended with LeNet: a compact CNN that turns image grids into class logits. Chapter 8 starts the modern CNN tour. AlexNet matters because it showed that a deeper CNN trained at large scale could learn useful visual representations directly from data instead of relying on hand-engineered features.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs. Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

## You are done when you can

- explain representation learning in the AlexNet story
- trace how an AlexNet-style stack reduces spatial resolution and expands channels
- explain why ReLU helped deeper networks compared with saturating activations
- explain why dropout appears in the dense classifier head
- debug a fixed-flatten-size failure when the input resolution changes


In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current

def conv2d_hw(height, width, kernel_size, stride=1, padding=0):
    h = math.floor((height + 2 * padding - kernel_size) / stride) + 1
    w = math.floor((width + 2 * padding - kernel_size) / stride) + 1
    return h, w

def pool2d_hw(height, width, kernel_size, stride):
    return conv2d_hw(height, width, kernel_size, stride, padding=0)


## 8.1.0 The Problem This Notebook Solves

LeNet already used convolution, nonlinear activation, pooling, flattening, and dense classification. AlexNet scales that idea in a historically important way:

- larger early receptive fields
- more channels
- more convolutional layers
- ReLU activations instead of saturating sigmoids
- dropout in the dense classifier head
- large-scale supervised training

Representation learning means the model learns intermediate features from data. Earlier computer vision systems often depended heavily on features written by humans, such as edge, texture, or shape descriptors. AlexNet-style CNNs learn many of those useful intermediate detectors as trainable parameters.

This notebook does not reproduce AlexNet's ImageNet training. That would require data, hardware, and time outside this chapter's purpose. The goal here is to make the architecture mechanically readable.


## 8.1.1 Spatial Size Shrinks While Channels Grow

Modern CNNs often follow a repeated pattern:

```text
spatial resolution goes down
channel count goes up
semantic richness goes up
```

Spatial resolution means height and width. Channel count means the number of feature maps at each spatial location. Early layers preserve more local detail; later layers store more abstract feature evidence across fewer locations.

Before running the cell, predict:

- The first large-stride convolution should reduce 96 by 96 sharply.
- Pooling should reduce spatial size again.
- Later convolutions with padding should preserve spatial size inside the stack.


In [ ]:
height, width = 96, 96
steps = [
    ("conv11 stride4 pad2", lambda h, w: conv2d_hw(h, w, 11, stride=4, padding=2)),
    ("pool3 stride2", lambda h, w: pool2d_hw(h, w, 3, stride=2)),
    ("conv5 pad2", lambda h, w: conv2d_hw(h, w, 5, padding=2)),
    ("pool3 stride2", lambda h, w: pool2d_hw(h, w, 3, stride=2)),
    ("conv3 pad1", lambda h, w: conv2d_hw(h, w, 3, padding=1)),
    ("conv3 pad1", lambda h, w: conv2d_hw(h, w, 3, padding=1)),
    ("conv3 pad1", lambda h, w: conv2d_hw(h, w, 3, padding=1)),
    ("pool3 stride2", lambda h, w: pool2d_hw(h, w, 3, stride=2)),
]

trace = []
for name, update in steps:
    height, width = update(height, width)
    trace.append((name, height, width))

print(trace)
assert trace[0][1:] == (23, 23)
assert trace[-1][1:] == (2, 2)


## 8.1.2 A Small AlexNet-Style Network

The real AlexNet used much larger channel counts and dense layers. This version keeps the same architectural pattern while reducing width so that the forward pass is quick:

```text
large early convolution -> pooling -> deeper conv stack -> pooling -> dense head
```

The dense head has a fixed input feature contract. For 96 by 96 inputs, the convolutional feature extractor below produces `(batch, 16, 2, 2)`, so flattening gives 64 features per example.

Before running the cell, predict:

- The output logits should have shape `(2, 10)`.
- The flattened size before the first linear layer should be 64.
- The model has trainable parameters in both the convolutional feature extractor and dense head.


In [ ]:
alexnet_small = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=11, stride=4, padding=2), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(8, 16, kernel_size=5, padding=2), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(16, 24, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(24, 24, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(24, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 2 * 2, 32), nn.ReLU(), nn.Dropout(p=0.5),
    nn.Linear(32, 32), nn.ReLU(), nn.Dropout(p=0.5),
    nn.Linear(32, 10),
)

X = torch.randn(2, 1, 96, 96)
rows, logits = trace_module_shapes(alexnet_small, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)
assert count_parameters(alexnet_small) > 0


## 8.1.3 ReLU Keeps a Stronger Gradient in the Positive Region

A saturating activation is an activation whose derivative becomes tiny over a wide input range. Sigmoid is useful in some places, but deep stacks of sigmoids can make gradient flow weak when activations saturate near 0 or 1.

ReLU is simple:

```text
relu(x) = max(0, x)
```

For positive inputs, its derivative is 1. That does not solve every optimization problem, but it makes deep networks easier to train than if every layer repeatedly squeezed values into a saturated range.

The cell compares gradients through a tiny activation-only computation. This is not a full training proof. It isolates one mechanical difference.


In [ ]:
values = torch.tensor([-6.0, -1.0, 0.0, 1.0, 6.0], requires_grad=True)
sigmoid_loss = torch.sigmoid(values).sum()
sigmoid_loss.backward()
sigmoid_grads = values.grad.clone()

values.grad.zero_()
relu_loss = F.relu(values).sum()
relu_loss.backward()
relu_grads = values.grad.clone()

print("sigmoid grads:", sigmoid_grads)
print("relu grads:", relu_grads)

assert relu_grads[-1].item() == 1.0
assert sigmoid_grads[-1].item() < 0.01


## 8.1.4 Dropout Changes Training Behavior, Not Evaluation Behavior

Dropout randomly zeros some activations during training. The purpose is regularization: it makes the dense classifier head less able to rely on one brittle co-adaptation of hidden units.

Two practical rules matter:

- `model.train()` enables dropout randomness.
- `model.eval()` disables dropout randomness and uses the full representation.

This is a software-state issue, not just a mathematical layer. A model that accidentally stays in training mode during inference can produce unstable predictions.


In [ ]:
drop = nn.Dropout(p=0.5)
X = torch.ones(12)

torch.manual_seed(1)
drop.train()
Y_train = drop(X)

drop.eval()
Y_eval = drop(X)

print("training output:", Y_train)
print("eval output:", Y_eval)

assert (Y_train == 0).any()
assert torch.equal(Y_eval, X)


## 8.1.5 Break It Deliberately: Fixed Flatten Size

The dense classifier sees a vector, not an image. If the convolutional stack produces a different spatial size, the flattened vector length changes. A fixed `Linear(in_features, out_features)` layer will then reject the input.

This is one of the most common CNN architecture mistakes:

```text
changed input resolution
changed convolution/pooling output size
forgot to update first dense layer
```

The cell intentionally feeds the small AlexNet a different input resolution and catches the failure so the notebook can continue.


In [ ]:
bad_input = torch.randn(2, 1, 80, 80)

try:
    alexnet_small(bad_input)
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected the dense layer to reject the changed flatten size")


## 8.1 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What does representation learning mean in the AlexNet story?
2. Why do modern CNNs often reduce spatial size while increasing channel count?
3. Why did ReLU help deeper CNNs compared with saturating activations?
4. What does dropout do differently in training and evaluation modes?
5. Why can changing image resolution break a fixed dense classifier head?
